# Olist E-Commerce Dataset — Exploratory Data Analysis

In this notebook we explore the structure of the Olist dataset 
before building the ETL pipeline.

## Objectives
- Understand the structure of each table
- Identify missing values
- Check referential integrity between tables
- Analyze the temporal coverage of the dataset

In [1]:
import pandas as pd
import os

# Ruta a los CSVs
RAW_PATH = "../data/raw/"

# Ver todos los archivos disponibles
archivos = os.listdir(RAW_PATH)
print("Archivos disponibles:")
for f in archivos:
    print(f" - {f}")

Archivos disponibles:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_orders_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


In [2]:
# Cargar todos los CSVs en un diccionario
tablas = {}
for archivo in archivos:
    nombre = archivo.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
    tablas[nombre] = pd.read_csv(RAW_PATH + archivo)

# Ver dimensiones de cada tabla
print(f"{'Tabla':<30} {'Filas':>10} {'Columnas':>10}")
print("-" * 52)
for nombre, df in tablas.items():
    print(f"{nombre:<30} {df.shape[0]:>10,} {df.shape[1]:>10}")

Tabla                               Filas   Columnas
----------------------------------------------------
customers                          99,441          5
geolocation                     1,000,163          5
orders                             99,441          8
order_items                       112,650          7
order_payments                    103,886          5
order_reviews                      99,224          7
products                           32,951          9
sellers                             3,095          4
product_category_name_translation         71          2


In [4]:
# Ver columnas y tipos de datos de cada tabla
for nombre, df in tablas.items():
    print(f"\n{'='*52}")
    print(f"  {nombre.upper()}")
    print(f"{'='*52}")
    print(df.dtypes)


  CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

  GEOLOCATION
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

  ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

  ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

  ORDER_PAYM

In [5]:
# Ver valores nulos por tabla
for nombre, df in tablas.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]  # Solo columnas con nulos
    if len(nulos) > 0:
        print(f"\n{'='*52}")
        print(f"  {nombre.upper()}")
        print(f"{'='*52}")
        print(nulos)


  ORDERS
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

  ORDER_REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

  PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [6]:
# ¿Los productos sin categoría tienen órdenes?
productos_sin_categoria = tablas['products'][tablas['products']['product_category_name'].isnull()]
print(f"Productos sin categoría: {len(productos_sin_categoria)}")

# Ver si aparecen en order_items
productos_sin_categoria_ids = productos_sin_categoria['product_id'].tolist()
ordenes_afectadas = tablas['order_items'][tablas['order_items']['product_id'].isin(productos_sin_categoria_ids)]
print(f"Órdenes que contienen esos productos: {len(ordenes_afectadas)}")

Productos sin categoría: 610
Órdenes que contienen esos productos: 1603


In [7]:
# Verificar integridad referencial entre tablas clave
# ¿Todos los order_items tienen una orden válida?
items_sin_orden = tablas['order_items'][~tablas['order_items']['order_id'].isin(tablas['orders']['order_id'])]
print(f"Order items sin orden válida: {len(items_sin_orden)}")

# ¿Todos los orders tienen un customer válido?
orders_sin_customer = tablas['orders'][~tablas['orders']['customer_id'].isin(tablas['customers']['customer_id'])]
print(f"Orders sin customer válido: {len(orders_sin_customer)}")

# ¿Todos los order_items tienen un producto válido?
items_sin_producto = tablas['order_items'][~tablas['order_items']['product_id'].isin(tablas['products']['product_id'])]
print(f"Order items sin producto válido: {len(items_sin_producto)}")

# ¿Todos los order_items tienen un seller válido?
items_sin_seller = tablas['order_items'][~tablas['order_items']['seller_id'].isin(tablas['sellers']['seller_id'])]
print(f"Order items sin seller válido: {len(items_sin_seller)}")

Order items sin orden válida: 0
Orders sin customer válido: 0
Order items sin producto válido: 0
Order items sin seller válido: 0


In [8]:
# Rango temporal del dataset
orders = tablas['orders'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

print("Rango temporal:")
print(f"  Desde: {orders['order_purchase_timestamp'].min()}")
print(f"  Hasta: {orders['order_purchase_timestamp'].max()}")

# Distribución de estados de órdenes
print("\nEstados de órdenes:")
print(orders['order_status'].value_counts())

Rango temporal:
  Desde: 2016-09-04 21:15:19
  Hasta: 2018-10-17 17:30:18

Estados de órdenes:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [9]:
# Vista previa de las tablas más importantes
tablas_clave = ['orders', 'order_items', 'products', 'customers', 'sellers', 'order_payments']

for nombre in tablas_clave:
    print(f"\n{'='*52}")
    print(f"  {nombre.upper()}")
    print(f"{'='*52}")
    print(tablas[nombre].head(3).to_string())


  ORDERS
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45           2018-08-13 00:00:00
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23          2018-08-08 13:50:00           2018-08-17 18:06:29           2018-09-04 00:00:00

  ORDER_ITEMS
                           order_id  order_item_id                        product_id                   